In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))


import pandas as pd
from src.conexionDB import getEngine, cargarDatosProcesados
from src.limpieza import limpiezaStrings, limpiarDocumento


In [2]:
motor = getEngine()

dataCruda = pd.read_sql(
    "SELECT * FROM datoscrudos.secop_contratos_2025",
    con=motor
)

dataCruda.shape

(1681982, 22)

In [3]:
columnasDateTime = ['fecha_de_firma_del_contrato', 'fecha_inicio_ejecuci_n', 'fecha_fin_ejecuci_n']
columnasNumeric = ['valor_contrato']
normalizacionTexto = ['estado_del_proceso','modalidad_de_contrataci_n', 'nivel_entidad', 'tipo_de_contrato', 'origen', 'nom_raz_social_contratista', 'tipo_documento_proveedor'  ]
strings = ['nivel_entidad', 'departamento_entidad', 'municipio_entidad', 'objeto_del_proceso', 'nom_raz_social_contratista', 'objeto_a_contratar', 'objeto_del_proceso']


for col in columnasDateTime:
    dataCruda[col] = pd.to_datetime(dataCruda[col], errors='coerce')

for col in columnasNumeric:
    dataCruda[col] = pd.to_numeric(dataCruda[col], errors='coerce')

for col in normalizacionTexto:
    dataCruda[col] = dataCruda[col].astype(str).str.strip().str.upper()

for col in strings:
    dataCruda[col] = dataCruda[col].astype(str).apply(limpiezaStrings)


In [4]:

dataCruda['nombre_de_la_entidad'] = dataCruda['nombre_de_la_entidad'].astype(str).apply(limpiezaStrings).str.upper()
dataCruda['documento_proveedor'] = dataCruda['documento_proveedor'].apply(limpiarDocumento)


In [5]:
dataCruda.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1681982 entries, 0 to 1681981
Data columns (total 22 columns):
 #   Column                       Non-Null Count    Dtype         
---  ------                       --------------    -----         
 0   nivel_entidad                1681982 non-null  object        
 1   codigo_entidad_en_secop      1681982 non-null  object        
 2   nombre_de_la_entidad         1681982 non-null  object        
 3   nit_de_la_entidad            1681982 non-null  object        
 4   departamento_entidad         1681982 non-null  object        
 5   municipio_entidad            1681982 non-null  object        
 6   estado_del_proceso           1681982 non-null  object        
 7   modalidad_de_contrataci_n    1681982 non-null  object        
 8   objeto_a_contratar           1681982 non-null  object        
 9   objeto_del_proceso           1681982 non-null  object        
 10  tipo_de_contrato             1681982 non-null  object        
 11  fecha_de_fi

In [6]:
dataCruda['estado_del_proceso'].value_counts(dropna=False)

estado_del_proceso
EN EJECUCIÓN              462336
MODIFICADO                438374
ACTIVO                    329835
CELEBRADO                 180671
APROBADO                  136486
TERMINADO                  69784
CERRADO                    40575
SUSPENDIDO                 11458
CEDIDO                     10046
LIQUIDADO                   1690
TERMINADO SIN LIQUIDAR       619
CONVOCADO                     90
BORRADOR                       5
EN APROBACIÓN                  5
ENVIADO PROVEEDOR              4
ADJUDICADO                     2
NO DEFINIDO                    1
CANCELADO                      1
Name: count, dtype: int64

In [7]:
dataCruda['modalidad_de_contrataci_n'].value_counts(dropna=False)

modalidad_de_contrataci_n
CONTRATACIÓN DIRECTA                                                                     1071401
CONTRATACIÓN RÉGIMEN ESPECIAL                                                             285988
CONTRATACIÓN DIRECTA (LEY 1150 DE 2007)                                                   125853
MÍNIMA CUANTÍA                                                                             68556
RÉGIMEN ESPECIAL                                                                           31770
SELECCIÓN ABREVIADA DE MENOR CUANTÍA                                                       16718
CONTRATACIÓN MÍNIMA CUANTÍA                                                                16670
CONTRATACIÓN DIRECTA (CON OFERTAS)                                                         15444
SELECCIÓN ABREVIADA SUBASTA INVERSA                                                        13657
CONTRATACIÓN RÉGIMEN ESPECIAL (CON OFERTAS)                                                13099
CONT

In [8]:
dataCruda['modalidad_de_contrataci_n'].isna().sum()

np.int64(0)

In [9]:
dataCruda['nivel_entidad'].value_counts(dropna=False)

nivel_entidad
TERRITORIAL             1226672
NACIONAL                 432620
CORPORACIÓN AUTÓNOMA      18801
NO DEFINIDO                3889
Name: count, dtype: int64

In [10]:
dataCruda['tipo_de_contrato'].value_counts(dropna=False)

tipo_de_contrato
PRESTACIÓN DE SERVICIOS           1438708
OTRO                                64383
DECRETO 092 DE 2017                 50195
SUMINISTROS                         34590
COMPRAVENTA                         28000
OBRA                                19602
SUMINISTRO                          11347
OTRO TIPO DE CONTRATO               10700
ARRENDAMIENTO DE INMUEBLES           7999
CONSULTORÍA                          4138
INTERVENTORÍA                        3523
SEGUROS                              2523
COMODATO                             2509
ARRENDAMIENTO                        1253
NO DEFINIDO                           663
ARRENDAMIENTO DE MUEBLES              551
SERVICIOS FINANCIEROS                 419
OPERACIONES DE CRÉDITO PÚBLICO        166
NO ESPECIFICADO                       152
CRÉDITO                               107
VENTA MUEBLES                         106
ACUERDO MARCO DE PRECIOS               85
CONCESIÓN                              55
ASOCIACIÓN PÚBLIC

In [11]:
dataCruda['origen'].value_counts(dropna=False)


origen
SECOPII    1498909
SECOPI      183073
Name: count, dtype: int64

In [12]:
dataCruda['duplicado_tecnico'] = dataCruda.duplicated()

len(dataCruda[dataCruda['duplicado_tecnico'] == True])


13358

In [13]:
dataCruda['flag_valor_valido'] = (
    dataCruda['valor_contrato'].notna() &
    (dataCruda['valor_contrato'] >= 0)
)

dataCruda['flag_valor_valido'].value_counts(normalize=True) * 100


flag_valor_valido
True    100.0
Name: proportion, dtype: float64

In [14]:
dataCruda['flag_fecha_firma_valida'] = (
    dataCruda['fecha_de_firma_del_contrato'].notna()
)

dataCruda['flag_fecha_firma_valida'].value_counts(normalize=True) * 100

flag_fecha_firma_valida
True    100.0
Name: proportion, dtype: float64

In [15]:
dataCruda['flag_inicio_posterior_firma'] = (
    dataCruda['fecha_inicio_ejecuci_n'].isna() |
    (dataCruda['fecha_inicio_ejecuci_n'] >= dataCruda['fecha_de_firma_del_contrato'])
)

dataCruda['flag_inicio_posterior_firma'].value_counts(normalize=True) * 100

flag_inicio_posterior_firma
True     99.98145
False     0.01855
Name: proportion, dtype: float64

In [16]:
len(dataCruda[dataCruda['flag_inicio_posterior_firma'] == False])


312

In [17]:
dataCruda['flag_fin_posterior_inicio'] = (
    dataCruda['fecha_fin_ejecuci_n'].isna() |
    (dataCruda['fecha_fin_ejecuci_n'] >= dataCruda['fecha_inicio_ejecuci_n'])
)


dataCruda['flag_fin_posterior_inicio'].value_counts(normalize=True) * 100

flag_fin_posterior_inicio
True     72.301428
False    27.698572
Name: proportion, dtype: float64

In [18]:
dataCruda['flag_fin_posterior_inicio'] = (
    dataCruda['fecha_inicio_ejecuci_n'].isna() |
    dataCruda['fecha_fin_ejecuci_n'].isna() |
    (dataCruda['fecha_fin_ejecuci_n'] >= dataCruda['fecha_inicio_ejecuci_n'])
)

dataCruda['flag_fin_posterior_inicio'].value_counts(normalize=True) * 100


flag_fin_posterior_inicio
True     99.972176
False     0.027824
Name: proportion, dtype: float64

In [19]:
len(dataCruda[dataCruda['flag_fin_posterior_inicio'] == False])

468

In [20]:
(
    dataCruda
    .loc[dataCruda['fecha_inicio_ejecuci_n'].isna(), 'estado_del_proceso']
    .value_counts()
)


estado_del_proceso
ACTIVO               329217
APROBADO             136220
EN APROBACIÓN             5
BORRADOR                  4
ENVIADO PROVEEDOR         3
Name: count, dtype: int64

In [21]:
dataCruda[dataCruda['estado_del_proceso'] == 'BORRADOR']

,nivel_entidad,codigo_entidad_en_secop,nombre_de_la_entidad,nit_de_la_entidad,departamento_entidad,municipio_entidad,estado_del_proceso,modalidad_de_contrataci_n,objeto_a_contratar,objeto_del_proceso,...,nom_raz_social_contratista,url_contrato,origen,tipo_documento_proveedor,documento_proveedor,duplicado_tecnico,flag_valor_valido,flag_fecha_firma_valida,flag_inicio_posterior_firma,flag_fin_posterior_inicio
409549,NACIONAL,702645490,SENA REGIONAL BOYACA GRUPO DE APOYO ADMINISTRA...,899999034,Boyacá,Sogamoso,BORRADOR,CONTRATACIÓN DIRECTA,Prestar los servicios personales para planear ...,Prestar los servicios personales para planear ...,...,ANDRES MAURICIO,https://community.secop.gov.co/Public/Tenderin...,SECOPII,CÉDULA DE CIUDADANÍA,1052407731,False,True,True,True,True
413448,NACIONAL,702988379,SERVICIO NACIONAL DE APRENDIZAJE - SENA MAGDALENA,8999990341,Magdalena,Santa Marta,BORRADOR,CONTRATACIÓN DIRECTA,Prestar servicios personales de carácter tempo...,Prestar servicios personales de carácter tempo...,...,ISABEL MARRIA OSORIO CASADIEGO,https://community.secop.gov.co/Public/Tenderin...,SECOPII,CÉDULA DE CIUDADANÍA,1082912437,False,True,True,True,True
419562,NACIONAL,704153634,SENA REGIONAL ATLANTICO INTERCENTROS,899999034,Atlántico,Barranquilla,BORRADOR,CONTRATACIÓN DIRECTA,Desarrollar formación profesional por competen...,Desarrollar formación profesional por competen...,...,KATIUSCA VANEGAS,https://community.secop.gov.co/Public/Tenderin...,SECOPII,CÉDULA DE CIUDADANÍA,22740768,False,True,True,True,True
434328,NACIONAL,702441627,SENA REGIONAL SANTANDER GRUPO DE APOYO ADMINIS...,899999034,Santander,Bucaramanga,BORRADOR,CONTRATACIÓN DIRECTA,Prestar servicios profesionales en la planeaci...,Prestar servicios profesionales en la planeaci...,...,LAURA JAZMIN VILLA CASTAÑO,https://community.secop.gov.co/Public/Tenderin...,SECOPII,CÉDULA DE CIUDADANÍA,1115193141,False,True,True,True,True
1579378,TERRITORIAL,268655011,SANTANDER - ALCALDÍA MUNICIPIO DE SABANA DE TO...,890204643,Santander,Sabana de Torres,BORRADOR,LICITACIÓN PÚBLICA,REMODELACION Y ADECUACION DE ESCENARIOS DEPORT...,REMODELACION Y ADECUACION DE ESCENARIOS DEPORT...,...,CH INGENIERIA Y CONSTRUCCION SAS,https://www.contratos.gov.co/consultas/detalle...,SECOPI,NIT DE PERSONA JURÍDICA,900911075,False,True,True,True,True


In [22]:
ESTADOS_REQUIEREN_INICIO = [
    'EN EJECUCIÓN',
    'SUSPENDIDO',
    'MODIFICADO',
    'CEDIDO',
    'TERMINADO',
    'TERMINADO SIN LIQUIDAR',
    'LIQUIDADO',
    'CERRADO'
]
dataCruda['flag_inicio_coherente_estado'] = (
    ~dataCruda['estado_del_proceso'].isin(ESTADOS_REQUIEREN_INICIO) |
    dataCruda['fecha_inicio_ejecuci_n'].notna()
)

dataCruda['flag_inicio_coherente_estado'].value_counts(normalize=True) * 100


flag_inicio_coherente_estado
True    100.0
Name: proportion, dtype: float64

In [23]:
ESTADOS_PRE_EJECUCION = [
    'BORRADOR',
    'EN APROBACIÓN',
    'APROBADO',
    'CONVOCADO',
    'ADJUDICADO',
    'ACTIVO',
    'ENVIADO PROVEEDOR',
    'CELEBRADO'
]


dataCruda['flag_inicio_inusual_para_estado'] = (    
    dataCruda['estado_del_proceso'].isin(ESTADOS_PRE_EJECUCION) &
    dataCruda['fecha_inicio_ejecuci_n'].notna()
)

dataCruda['flag_inicio_inusual_para_estado'].value_counts(normalize=True) * 100

flag_inicio_inusual_para_estado
False    89.200301
True     10.799699
Name: proportion, dtype: float64

In [24]:
dataCruda['flag_claves_presentes'] = (
    dataCruda['numero_del_contrato'].notna() &
    dataCruda['numero_de_proceso'].notna()
)

dataCruda['flag_claves_presentes'].value_counts(normalize=True) * 100


flag_claves_presentes
True    100.0
Name: proportion, dtype: float64

In [25]:
dataCruda['flag_registro_valido'] = (
    dataCruda['flag_valor_valido'] &
    dataCruda['flag_fecha_firma_valida'] &
    dataCruda['flag_inicio_posterior_firma'] &
    dataCruda['flag_fin_posterior_inicio'] &
    dataCruda['flag_claves_presentes'] &
    dataCruda['flag_inicio_coherente_estado'] &
    ~dataCruda['duplicado_tecnico']
)


dataCruda['flag_registro_valido'].value_counts(normalize=True) * 100

flag_registro_valido
True     99.159503
False     0.840497
Name: proportion, dtype: float64

In [26]:
dataCruda = dataCruda.reset_index(drop=True)
dataCruda['id_registro_raw'] = dataCruda.index

quality_log = []


In [27]:
mask = ~dataCruda['flag_valor_valido']

for _, row in dataCruda.loc[mask].iterrows():
    quality_log.append({
        'id_registro_raw': row['id_registro_raw'],
        'tipo': 'ERROR',
        'regla': 'VALOR_CONTRATO_INVALIDO',
        'descripcion': 'El valor del contrato es nulo, cero o negativo',
        'severidad': 'CRITICA'
    })

mask = ~dataCruda['flag_fecha_firma_valida']

for _, row in dataCruda.loc[mask].iterrows():
    quality_log.append({
        'id_registro_raw': row['id_registro_raw'],
        'tipo': 'ERROR',
        'regla': 'FECHA_FIRMA_NULA',
        'descripcion': 'El contrato no tiene fecha de firma informada',
        'severidad': 'CRITICA'
    })

mask = ~dataCruda['flag_inicio_posterior_firma']

for _, row in dataCruda.loc[mask].iterrows():
    quality_log.append({
        'id_registro_raw': row['id_registro_raw'],
        'tipo': 'ERROR',
        'regla': 'INICIO_ANTES_DE_FIRMA',
        'descripcion': 'La fecha de inicio es anterior a la fecha de firma del contrato',
        'severidad': 'CRITICA'
    })

mask = ~dataCruda['flag_fin_posterior_inicio']

for _, row in dataCruda.loc[mask].iterrows():
    quality_log.append({
        'id_registro_raw': row['id_registro_raw'],
        'tipo': 'ERROR',
        'regla': 'FIN_ANTES_DE_INICIO',
        'descripcion': 'La fecha de fin es anterior a la fecha de inicio',
        'severidad': 'CRITICA'
    })

mask = ~dataCruda['flag_claves_presentes']

for _, row in dataCruda.loc[mask].iterrows():
    quality_log.append({
        'id_registro_raw': row['id_registro_raw'],
        'tipo': 'ERROR',
        'regla': 'CLAVES_CONTRATO_FALTANTES',
        'descripcion': 'El contrato no tiene número de contrato o número de proceso',
        'severidad': 'CRITICA'
    })


mask = ~dataCruda['flag_inicio_coherente_estado']

for _, row in dataCruda.loc[mask].iterrows():
    quality_log.append({
        'id_registro_raw': row['id_registro_raw'],
        'tipo': 'ERROR',
        'regla': 'INICIO_REQUERIDO_POR_ESTADO',
        'descripcion': 'El estado del contrato requiere fecha de inicio y no está informada',
        'severidad': 'CRITICA'
    })

mask = dataCruda['flag_inicio_inusual_para_estado']

for _, row in dataCruda.loc[mask].iterrows():
    quality_log.append({
        'id_registro_raw': row['id_registro_raw'],
        'tipo': 'ALERTA',
        'regla': 'INICIO_INUSUAL_PARA_ESTADO',
        'descripcion': 'Fecha de inicio presente en un estado previo a la ejecución',
        'severidad': 'LEVE'
    })



In [28]:
df_quality_log = pd.DataFrame(quality_log)

In [29]:
df_quality_log['severidad'].value_counts()


severidad
LEVE       181649
CRITICA       780
Name: count, dtype: int64

In [30]:
df_quality_log['regla'].value_counts()


regla
INICIO_INUSUAL_PARA_ESTADO    181649
FIN_ANTES_DE_INICIO              468
INICIO_ANTES_DE_FIRMA            312
Name: count, dtype: int64

In [31]:
df_quality_log['created_at'] = pd.Timestamp.utcnow()


In [32]:
motor = getEngine()

cargarDatosProcesados(df_quality_log, motor, 'data_quality_log')

In [33]:
df_validos = dataCruda[dataCruda['flag_registro_valido']].copy()
len(df_validos)

1667845

In [34]:
columnas_a_excluir = [
    'flag_valor_valido',
    'flag_fecha_firma_valida',
    'flag_inicio_posterior_firma',
    'flag_fin_posterior_inicio',
    'flag_claves_presentes',
    'flag_inicio_coherente_estado',
    'flag_inicio_inusual_para_estado',
    'flag_registro_valido',
    'duplicado_tecnico'
    
]

df_processed = df_validos.drop(columns=columnas_a_excluir, errors='ignore')


In [35]:
motor = getEngine()

cargarDatosProcesados(df_processed, motor, 'secop_contratos_2025_procesados_historico')

In [36]:
df_processed['duracion_dias'] = (
    df_processed['fecha_fin_ejecuci_n'] - df_processed['fecha_inicio_ejecuci_n']
).dt.days


In [37]:
PRIORIDAD_ESTADO = {
    'TERMINADO': 4,
    'LIQUIDADO': 4,
    'CERRADO': 4,

    'MODIFICADO': 3,
    'CEDIDO': 3,
    'SUSPENDIDO': 3,

    'EN EJECUCIÓN': 2,

    'ACTIVO': 1,
    'APROBADO': 1,
    'BORRADOR': 1,
    'EN APROBACIÓN': 1,
    'CONVOCADO': 1,
    'ADJUDICADO': 1,
    'ENVIADO PROVEEDOR': 1
}

CLAVE_CONTRATO = [
    'numero_del_contrato',
    'documento_proveedor',
    'codigo_entidad_en_secop'
]


In [38]:
df_processed_financiero = (
    df_processed
    .assign(
        prioridad=df_processed['estado_del_proceso']
            .map(PRIORIDAD_ESTADO)
            .fillna(0),

        duracion_dias=(
            df_processed['fecha_fin_ejecuci_n'] -
            df_processed['fecha_inicio_ejecuci_n']
        ).dt.days
    )
    .sort_values(
        CLAVE_CONTRATO + [
            'prioridad',
            'fecha_inicio_ejecuci_n',
            'duracion_dias',
            'fecha_de_firma_del_contrato'
        ],
        ascending=[
            True, True, True,   
            False,              
            False,              
            False,              
            False               
        ]
    )
    .drop_duplicates(
        subset=CLAVE_CONTRATO,
        keep='first'
    )
)


In [39]:
columnas_a_excluir = [
    'duracion_dias',
    'prioridad'
    
]

df_processed = df_processed_financiero.drop(columns=columnas_a_excluir, errors='ignore')

In [40]:
motor = getEngine()

cargarDatosProcesados(df_processed_financiero, motor, 'secop_contratos_2025_procesados_financiero')